# Week 8 · Day 1 — Product analytics I: starter

Nur Finance · Kamola's questions: where do we lose people in the app, and do borrowers come back?
Run the setup, then write **two functions** (TODO) until their assert cells pass, then one cell per practice task.
SQL or pandas — your choice per task; `sql("...")` runs a query on `nur.db`.

In [1]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
DATA = Path("../..")                      # this notebook lives in data/python/week8/, so "../.." is the data/ folder
con = sqlite3.connect(DATA / "raw" / "nur.db")
ASOF = "2026-10-01"
sql = lambda q: pd.read_sql(q, con)       # sql("SELECT ...") returns a DataFrame
print(sql("SELECT COUNT(*) AS loans FROM loans"))

   loans
0   7828


## 0. The tables you need today

In [2]:
events = sql("SELECT * FROM app_events")
apps = sql("SELECT application_id, customer_id, channel, status, applied_at FROM applications")
loans = sql("SELECT loan_id, application_id, customer_id, product_id, disbursed_at, principal FROM loans")
print(len(events), "events ·", events["application_id"].nunique(), "sessions ·", len(apps), "applications ·", len(loans), "loans")
events.head()

22784 events · 5006 sessions · 14000 applications · 7828 loans


,event_id,customer_id,event_name,event_time,application_id,device,app_version
0,1,698,app_open,2025-01-12 08:08:00,6,android,3.9.1
1,2,698,view_offer,2025-01-12 08:33:00,6,android,3.8.0
2,3,698,start_application,2025-01-12 09:03:00,6,android,3.8.0
3,4,1397,app_open,2026-03-26 12:44:00,7,android,3.9.1
4,5,1397,view_offer,2026-03-26 13:11:00,7,android,4.0.0


## 1. TODO — `session_table(events, loans)`
One row per session (`application_id`), a 0/1 column per step in this order:
`app_open, view_offer, start_application, submit_application, approved, contract_signed`, plus `disbursed`
(1 if the application has a row in `loans`), plus `device` and `app_version` **taken from the session's `app_open` event**.

In [3]:
STEPS = ["app_open", "view_offer", "start_application", "submit_application", "approved", "contract_signed"]

def session_table(events, loans):
    """DataFrame indexed by application_id: one 0/1 column per step in STEPS, 'disbursed', 'device', 'app_version'."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [4]:
s = session_table(events, loans)
assert s.index.is_unique and len(s) == events["application_id"].nunique()
assert list(s.columns[:7]) == STEPS + ["disbursed"]
assert set(s[STEPS + ["disbursed"]].stack().unique()) <= {0, 1}
assert (s["contract_signed"] <= s["approved"]).all() and (s["disbursed"] <= s["contract_signed"]).all()
first = events.sort_values("event_time").groupby("application_id").first()
assert (s.loc[first.index, "device"] == first["device"]).all(), "device must come from the app_open (first) event"
print("session_table OK")

NotImplementedError: TODO: write me, then run the assert cell below

## 2. TODO — `repeat_table(loans, months=(3, 6, 12), asof=ASOF)`
One row per first-loan month (`YYYY-MM`): `customers` and, per N in `months`, the % of customers whose **second** loan was
disbursed within N months of the first (`first + pd.DateOffset(months=N)`). A cell must be **NaN** unless every first loan
of that cohort is at least N months old on the as-of date.

In [5]:
def repeat_table(loans, months=(3, 6, 12), asof=ASOF):
    """DataFrame indexed by cohort 'YYYY-MM': 'customers', then 'm3', 'm6', 'm12' (percent, NaN when not observable)."""
    raise NotImplementedError("TODO: write me, then run the assert cell below")

In [6]:
r = repeat_table(loans)
assert r["customers"].sum() == loans["customer_id"].nunique()
assert np.isnan(r.loc["2026-09", "m3"]) and np.isnan(r.loc["2025-12", "m12"]), "unobservable cells must be blank"
assert (r["m6"].dropna() >= r.loc[r["m6"].notna(), "m3"]).all(), "within 6 months includes within 3"
print("repeat_table OK")

NotImplementedError: TODO: write me, then run the assert cell below

## P1

The metric tree for September 2026 (the worked example's definitions) — each factor, and the product.

In [7]:
# your code

## P2

August vs September: factor changes and their shares of the log change; is take-up's fall real?

In [8]:
# your code

## P3

The app funnel: count, step conversion, conversion from the top, drop-off; add disbursed.

In [9]:
# your code

## P4

All applications left-joined to sessions: share with an app_open by channel. Then the funnel every channel has (applications + loans).

In [10]:
# your code

## P5

Is device constant inside a session? Funnel by event-level device vs by session device; then by version.

In [11]:
# your code

## P6

Approval in the funnel vs approval in the report; the cross-tab of status × has-a-submit-event.

In [12]:
# your code

## P7

Signed but no loan: how many, by status, and which are only waiting on the as-of date.

In [13]:
# your code

## P8

repeat_table: the earliest full cohort, and the cells that must be blank.

In [14]:
# your code

## P9

The triangle by quarter at 1, 2, 3, 6, 9, 12, 18, 24 months.

In [15]:
# your code

## P10

DAU, MAU and stickiness for the last 12 months.

In [16]:
# your code

## P11 (stretch)

Repeat borrowing by the product of the first loan; the product of the second loan.

In [17]:
# your code